In [9]:
# conda activate genomic_tools

import os
import json
import pickle
import pandas as pd
from collections import defaultdict

## Load interproscan results

In [2]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [3]:
# Parse the PIRSR data

with open("data/interproscan/interpro/data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [4]:
interproscan_results = interproscan_results.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

## Load event info.

In [5]:
# # load splicing event info.
# import pickle

# # note: this dictionary includes ALL events that were emitted, not just significant ones.
# with open('data/event_info.pkl', 'rb') as f:
#     event_info = pickle.load(f)
    
# # # note: this dictionary includes ALL events that were emitted, not just significant ones.
# # with open('data/event_dicts.pkl', 'rb') as f:
# #     event_dicts = pickle.load(f)

In [6]:
# if an event is cell type-specific, and it has a exon_diff_boundary orexon_diff_junction transcript, I need to know if:

# (1) the transcript was emitted and 
# (2) it is also cell type-specific.

# this is important because it means that any overlapping domains of interest are not necessarily cell type-specific

## Map events to interproscan results

(Get the transcript associated with each significant event)

In [7]:
# get all significant splicing events

signif_events = []
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        for idx, _ in signif_exons_df.iterrows():
            if idx not in signif_events:
                signif_events.append(idx)

In [10]:
# map events to interproscan results

with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

columns = ['protein_accession', 'sequence_length', 'analysis', 
           'signature_description', 'start', 'stop', 'interpro_description']
analyses_to_exclude = ['NCBIFAM', 'SFLD'] # these tools are for full-length protein classification so it doesn't make sense to consider them as overlapping with a single exon
ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

def overlaps_skip(df, aa_start):
    return df[(df['start'] <= aa_start) & (df['stop'] >= aa_start)]

# index ipr by protein_accession once
ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

event_interproscan_map = defaultdict(dict)
  
for ev, rec in event_protein_map.items():
    
    # inclusion: direct lookup instead of boolean mask
    rec_incl = rec['inclusion']
    incl_df = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    overlap_df = incl_df[(incl_df['start'] <= rec_incl['aa_end']) & 
                         (incl_df['stop']  >= rec_incl['aa_start'])]
    if overlap_df.empty:
        continue
    event_interproscan_map[ev]['inclusion'] = overlap_df.assign(
        aa_start = rec_incl['aa_start'],
        aa_end = rec_incl['aa_end'],
        exon_cds_start = rec_incl['exon_cds_start'],
        exon_cds_end = rec_incl['exon_cds_end'],
        frame_preserving = rec_incl['frame_preserving'],
        clean_start = rec_incl['clean_start'],
        clean_end = rec_incl['clean_end'],
    ).reset_index(drop=True)

    # real skip
    if rec.get('real_skip'):
        skip_df = ipr_grouped.get(rec['real_skip'])
        if skip_df is not None:
            s = overlaps_skip(skip_df, rec_incl['aa_start'])
            if not s.empty:
                event_interproscan_map[ev]['real_skip'] = s.reset_index(drop=True)

    # synthetic skip
    if rec.get('synthetic_skip'):
        synth_df = ipr_grouped.get(rec['synthetic_skip'])
        if synth_df is not None:
            s = overlaps_skip(synth_df, rec_incl['aa_start'])
            if not s.empty:
                event_interproscan_map[ev]['synthetic_skip'] = s.reset_index(drop=True)

    # junction siblings
    if rec.get('exon_diff_junction_siblings'):
        rec_sib = rec['exon_diff_junction_siblings']
        frames = [ipr_grouped[sib['transcript_id']] 
                  for sib in rec['exon_diff_junction_siblings']
                  if sib['transcript_id'] in ipr_grouped]
        if frames:
            event_interproscan_map[ev]['exon_diff_junction_siblings'] = pd.concat(frames, ignore_index=True)

    # boundary siblings
    if rec.get('exon_diff_boundary_siblings'):
        rec_sib = rec['exon_diff_boundary_siblings']
        frames = [ipr_grouped[sib['transcript_id']] 
                  for sib in rec['exon_diff_boundary_siblings']
                  if sib['transcript_id'] in ipr_grouped]
        if frames:
            event_interproscan_map[ev]['exon_diff_boundary_siblings'] = pd.concat(frames, ignore_index=True)

In [11]:
with open('data/event_interproscan_map.pkl', "wb") as file:
    pickle.dump(event_interproscan_map, file)

## Merge interproscan results with significant splicing event info.

In [ ]:
# merge cell type-specific events with InterProScan results

df_list = dict() 

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        event_interproscan_dict = {ev: event_interproscan_map[ev] for ev in signif_events_df.index if ev in event_interproscan_map}

        result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
            for ev, buckets in event_interproscan_dict.items()
            for bucket, df in buckets.items()],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in result.columns if c not in ('event_id', 'bucket')]
        result = result[cols]
 
        # restrict to splicing events that overlap with protein domains identified by InterProScan
        df = result.merge(signif_events_df, left_on="event_id", right_index=True)
 
        # summarize interpro results for significant splicing events
        df_list[ctype] = df.groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            interproscan_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            chr=('chr', lambda x: ' | '.join(x.unique())),
            exon_start=('exon_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_end=('exon_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_len=('exon_len', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_start=('exon_cds_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_end=('exon_cds_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_start=('aa_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_end=('aa_end', lambda x: ' | '.join(map(str, x.unique()))),
            domain_start=('start', lambda x: ' | '.join(map(str, x.unique()))),
            domain_stop=('stop', lambda x: ' | '.join(map(str, x.unique()))),
            protein_sequence_length=('sequence_length', lambda x: ' | '.join(map(str, x.unique()))),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(x.unique())),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [14]:
with open("data/signif_events_interproscan_summary.pkl", "wb") as file:
    pickle.dump(df_list, file)

In [15]:
# load splicing event info.
# note: this dictionary includes ALL events that were emitted, not just significant ones.

with open('data/event_info.pkl', 'rb') as f:
    event_info = pickle.load(f)

In [32]:
event_info['ENSG00000116983_ProteinCoding_1']

{'meta': {'chrom': 'chr1',
  'strand': '-',
  'es': 39683937,
  'ee': 39684152,
  'gene': 'ENSG00000116983',
  'us_intron_start': 39682734,
  'ds_intron_end': 39684441},
 'cluster_id': 785,
 'compatible': {'ENST00000372844': {'aa_start': 54,
   'aa_end': 125,
   'coding_nt_length': 216,
   'overlap_type': 'fully_coding',
   'gtf_frame': 0,
   'clean_start': True,
   'clean_end': True,
   'frame_preserving': True,
   'exon_cds_start': 39683937,
   'exon_cds_end': 39684152,
   'transcript_type': 'protein_coding',
   'transcript_tag': 'basic,Ensembl_canonical,GENCODE_Primary,MANE_Select,appris_principal_1,CCDS'},
  'ENST00000617690': {'aa_start': 54,
   'aa_end': 125,
   'coding_nt_length': 216,
   'overlap_type': 'fully_coding',
   'gtf_frame': 0,
   'clean_start': True,
   'clean_end': True,
   'frame_preserving': True,
   'exon_cds_start': 39683937,
   'exon_cds_end': 39684152,
   'transcript_type': 'protein_coding',
   'transcript_tag': 'basic,appris_principal_1,CCDS'}},
 'exon_diff_j

In [27]:
df_list['All_GABAergic'][df_list['All_GABAergic']['is_specific'] == "True"].sort_values('n_analyses', ascending=False).head(10)

,event_id,bucket,r,is_specific,Gene,transcript_id,chr,exon_start,exon_end,exon_len,...,exon_aa_start,exon_aa_end,domain_start,domain_stop,protein_sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
2023,ENSG00000092096_ProteinCoding_2,inclusion,0.2085403902319068,True,SLC22A17,ENST00000354772,chr14,23351752,23351855,104,...,200.0,234.0,177 | 150 | 174 | 217 | 207 | 230 | 1 | 215 | ...,599 | 587 | 597 | 229 | 240 | 206 | 596 | 245 ...,631,False,10,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,MFS general substrate transporter like domains...,MFS transporter superfamily | - | Major facili...
2185,ENSG00000099204_ProteinCoding_2,exon_diff_boundary_siblings,-0.3991881867582086,True,ABLIM1,ENST00000533213,chr10,114447880,114448026,147,...,nan,nan,216 | 284 | 88 | 152 | 702 | 86 | 700 | 217 | ...,283 | 349 | 151 | 215 | 778 | 150 | 216 | 350 ...,778,nan,10,CATH-Gene3D | CATH-FunFam | CDD | MobiDB-lite ...,Cysteine Rich Protein | Villin headpiece domai...,- | Villin headpiece domain superfamily | Puta...
595,ENSG00000049323_ProteinCoding_2,inclusion,-0.2017625375727214,True,LTBP1,ENST00000404816,chr2,33342838,33342963,126,...,1243.0,1285.0,1203 | 1244 | 1202 | 872 | 1201 | 1285 | 1224 ...,1243 | 1286 | 1328 | 1276 | 1314 | 1284 | 1316...,1721,True,10,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Laminin | latent-transforming growth factor be...,- | Complement Clr-like EGF domain | EGF-like ...
10286,ENSG00000179520_ProteinCoding_1,inclusion,0.3529908454564064,True,SLC17A8,ENST00000323346,chr12,100402596,100402745,150,...,301.0,350.0,310 | 79 | 117 | 271 | 334 | 345 | 73 | 75 | 315,508 | 502 | 463 | 309 | 344 | 333 | 370 | 503 ...,589,True,9,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,MFS general substrate transporter like domains...,MFS transporter superfamily | - | Major facili...
1960,ENSG00000090621_ProteinCoding_3,exon_diff_boundary_siblings,0.1962318802688724,True,PABPC4,ENST00000678625,chr1,39564686,39564773,88,...,nan,nan,489 | 17 | 111 | 207 | 206 | 110 | 22 | 195 | ...,583 | 110 | 206 | 312 | 311 | 205 | 108 | 215 ...,584,nan,9,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,c-terminal domain of poly(a) binding protein |...,- | Nucleotide-binding alpha-beta plait domain...
4934,ENSG00000128342_ProteinCoding_1,inclusion,-0.239586841772764,True,LIF,ENST00000249075,chr22,30244755,30244933,179,...,6.0,65.0,23 | 43 | 3 | 33 | 15 | 1 | 34 | 24,202 | 14 | 32 | 191 | 22,202,False,9,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,- | leukemia inhibitory factor | LIF / OSM fam...,"Four-helical cytokine-like, core | - | Leukemi..."
6940,ENSG00000144355_ProteinCoding_1,inclusion,-0.2422743850073355,True,DLX1,ENST00000361725,chr2,172086654,172086853,200,...,104.0,170.0,123 | 129 | 95 | 100 | 128 | 125 | 161 | 126,189 | 185 | 118 | 112 | 190 | 184 | 186,255,False,9,CATH-Gene3D | CATH-FunFam | CDD | MobiDB-lite ...,Homeodomain-like | Distal-less homeobox 1 | - ...,- | Homeodomain | Homeodomain-like superfamily...
5902,ENSG00000136717_ProteinCoding_6,exon_diff_junction_siblings,-0.2212741177486916,True,BIN1,ENST00000393040 | ENST00000346226,chr2,127057473,127057601,129,...,nan,nan,10 | 400 | 157 | 222 | 31 | 410 | 249 | 342 | ...,245 | 482 | 191 | 242 | 241 | 481 | 323 | 377 ...,482 | 518,nan,9,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Arfaptin homology (AH) domain/BAR domain | SH3...,"AH/BAR domain superfamily | - | Amphiphysin 2,..."
3976,ENSG00000116337_ProteinCoding_2,exon_diff_junction_siblings,-0.4510693963749933,True,AMPD2,ENST00000528667 | ENST00000528454,chr1,109625303,109625433,131,...,nan,nan,270 | 371 | 400 | 99 | 301 | 1 | 11 | 55 | 163...,796 | 461 | 119 | 797 | 49 | 21 | 814 | 806 | ...,825 | 761,nan,9,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Metal-dependent hydrolases | - | AMP deaminase...,- | AMP deaminase | Metal-dependent hydrolase ...
4032,ENSG00000116983_ProteinCoding_1,inclusion,-0.2819875540122913,True,HPCAL4,ENST00000372844,chr1,3968393

In [29]:
for ct, df in df_list.items():
    if df['transcript_id'].isin(["ENST00000556803", "ENST00000556803"]).any():
        print(ct)

In [250]:
df_list['Deep_layer_glutamatergic'][df_list['Deep_layer_glutamatergic']['is_specific'] == "True"].sort_values('n_analyses', ascending=False).head(10)

,event_id,bucket,r,is_specific,Gene,transcript_id,chr,exon_start,exon_end,exon_len,...,exon_aa_start,exon_aa_end,domain_start,domain_stop,protein_sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
481,ENSG00000056291_ProteinCoding_1,inclusion,0.4471791265064418,True,NPFFR2,ENST00000308744,chr4,72128585,72128919,335,...,0.0,109.0,20 | 46 | 33 | 62 | 44 | 82 | 71 | 1 | 104 | 5...,388 | 386 | 344 | 343 | 333 | 70 | 103 | 81 | ...,420,False,11,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Rhodopsin 7-helix transmembrane proteins | neu...,"- | G protein-coupled receptor, rhodopsin-like..."
756,ENSG00000071054_ProteinCoding_3,exon_diff_junction_siblings,0.4690755503915918,True,MAP4K4,ENST00000324219,chr2,101863821,101864051,231,...,nan,nan,108 | 1 | 359 | 384 | 507 | 306 | 317 | 421 | ...,314 | 107 | 379 | 492 | 527 | 349 | 338 | 462 ...,1384,nan,10,CATH-Gene3D | CATH-FunFam | COILS | MobiDB-lit...,Transferase(Phosphotransferase) domain 1 | Pho...,- | Citron homology (CNH) domain | Protein kin...
6465,ENSG00000177508_ProteinCoding_1,inclusion,-0.1988403026764951,True,IRX3,ENST00000329734,chr16,54284497,54285613,1117,...,89.0,461.0,128 | 129 | 209 | 135 | 190 | 210 | 227 | 324 ...,188 | 232 | 189 | 381 | 220 | 258 | 339 | 184 ...,501,False,10,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Homeodomain-like | Iroquois-class homeobox pro...,- | Homeodomain | KN homeodomain | Iroquois-cl...
2114,ENSG00000108387_ProteinCoding_3,exon_diff_junction_siblings,0.3501897618647627,True,SEPTIN4,ENST00000317268,chr17,58526682,58526978,297,...,nan,nan,120 | 121 | 453 | 141 | 1 | 13 | 95 | 428 | 143,416 | 476 | 415 | 115 | 26 | 108 | 448 | 420 |...,478,nan,9,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,P-loop containing nucleotide triphosphate hydr...,P-loop containing nucleoside triphosphate hydr...
4603,ENSG00000146904_ProteinCoding_1,inclusion,0.3414524560565585,True,EPHA1,ENST00000275815,chr7,143394808,143395014,207,...,715.0,783.0,706 | 629 | 674 | 604 | 729 | 627 | 625 | 628 ...,900 | 894 | 835 | 885 | 887 | 827 | 829 | 763 ...,976,True,9,CATH-Gene3D | CATH-FunFam | PIRSR | Pfam | Pho...,Transferase(Phosphotransferase) domain 1 | Rec...,"- | Serine-threonine/tyrosine-protein kinase, ..."
188,ENSG00000010704_ProteinCoding_7,inclusion,-0.1190834969825156,True,HFE,ENST00000353147,chr6,26092685,26092960,276,...,25.0,117.0,26 | 36 | 21 | 40 | 30 | 100 | 7,115 | 109 | 124 | 112 | 119 | 106,168,True,8,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,Immunoglobulins | Major histocompatibility com...,Immunoglobulin-like fold | - | Immunoglobulin ...
4118,ENSG00000139880_ProteinCoding_1,real_skip,-0.4879409185382066,True,CDH24,ENST00000487137,chr14,23051970,23052083,114,...,nan,nan,371 | 379 | 30 | 396 | 363 | 375,476 | 477 | 475 | 468 | 602 | 486 | 479,781,nan,8,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Cadherins | Cadherin 24 | Cadherin tandem repe...,- | Cadherin-like | Cadherin-like superfamily
4117,ENSG00000139880_ProteinCoding_1,inclusion,-0.4879409185382066,True,CDH24,ENST00000397359,chr14,23051970,23052083,114,...,454.0,492.0,371 | 379 | 30 | 396 | 375,515 | 469 | 513 | 454 | 640 | 458 | 517,819,True,8,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Cadherins | Protocadherin beta 4 | Cadherin ta...,- | Cadherin-like | Cadherin-like superfamily
4119,ENSG00000139880_ProteinCoding_1,synthetic_skip,-0.4879409185382066,True,CDH24,ENSG00000139880_ProteinCoding_1_synthetic_skip,chr14,23051970,23052083,114,...,nan,nan,371 | 379 | 30 | 396 | 363 | 375,476 | 477 | 475 | 468 | 602 | 486 | 479,781,nan,8,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Cadherins | Cadherin 24 | Cadherin tandem repe...,- | Cadherin-like | Cadherin-like superfamily
4882,ENSG00000151914_ProteinCoding_4,inclusion,-0.525019250063025,True,DST,ENST00000680361,chr6,56529448,56529774,327,...,5756.0,5864.0,5715 | 5824 | 5825 | 5607 | 5714 | 31 | 5705,5823 | 5933 | 5824 | 5820 | 5929 | 5817 | 7818...,7818,True,7,CAT

In [ ]:
event_info['ENSG00000177508_ProteinCoding_1']

{'meta': {'chrom': 'chr4',
  'strand': '+',
  'es': 72128585,
  'ee': 72128919,
  'gene': 'ENSG00000056291',
  'us_intron_start': 72032201,
  'ds_intron_end': 72138039},
 'cluster_id': 9150,
 'compatible': {'ENST00000308744': {'aa_start': 0,
   'aa_end': 109,
   'coding_nt_length': 328,
   'overlap_type': 'partially_coding',
   'gtf_frame': 0,
   'clean_start': True,
   'clean_end': False,
   'frame_preserving': False,
   'exon_cds_start': 72128592,
   'exon_cds_end': 72128919,
   'transcript_type': 'protein_coding',
   'transcript_tag': 'upstream_ATG,basic,Ensembl_canonical,GENCODE_Primary,MANE_Select,appris_principal_4,CCDS'}},
 'exon_diff_junction': {'ENST00000395999': {'aa_start': 0,
   'aa_end': 112,
   'coding_nt_length': 335,
   'overlap_type': 'fully_coding',
   'gtf_frame': 1,
   'clean_start': False,
   'clean_end': False,
   'frame_preserving': False,
   'exon_cds_start': 72128585,
   'exon_cds_end': 72128919,
   'transcript_type': 'protein_coding',
   'transcript_tag': 'bas

In [ ]:
df_list['Deep_layer_glutamatergic'][df_list['Deep_layer_glutamatergic']['is_specific'] == "True"].sort_values('n_analyses', ascending=False).head(10)

,event_id,r,is_specific,Gene,transcript_id,chr,exon_start,exon_end,exon_len,coding_nt_length,sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
228,ENSG00000056291_ProteinCoding_1,0.4471791265064418,True,NPFFR2,ENST00000308744,chr4,72128585,72128919,335,328,420,False,11,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Rhodopsin 7-helix transmembrane proteins | neu...,"- | G protein-coupled receptor, rhodopsin-like..."
3121,ENSG00000177508_ProteinCoding_1,-0.1988403026764951,True,IRX3,ENST00000329734,chr16,54284497,54285613,1117,1117,501,False,10,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Homeodomain-like | Iroquois-class homeobox pro...,- | Homeodomain | KN homeodomain | Iroquois-cl...
2207,ENSG00000146904_ProteinCoding_1,0.3414524560565585,True,EPHA1,ENST00000275815,chr7,143394808,143395014,207,207,976,True,9,CATH-Gene3D | CATH-FunFam | PIRSR | Pfam | Pho...,Transferase(Phosphotransferase) domain 1 | Rec...,"- | Serine-threonine/tyrosine-protein kinase, ..."
1976,ENSG00000139880_ProteinCoding_1,-0.4879409185382066,True,CDH24,ENST00000397359,chr14,23051970,23052083,114,114,819,True,8,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Cadherins | Protocadherin beta 4 | Cadherin ta...,- | Cadherin-like | Cadherin-like superfamily
87,ENSG00000010704_ProteinCoding_7,-0.1190834969825156,True,HFE,ENST00000353147,chr6,26092685,26092960,276,276,168,True,8,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,Immunoglobulins | Major histocompatibility com...,Immunoglobulin-like fold | - | Immunoglobulin ...
1198,ENSG00000115593_ProteinCoding_1,0.2891061506351097,True,SMYD1,ENST00000419482,chr2,88093517,88093555,39,39,490,True,7,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SMART...,SET domain | Histone-lysine N-methyltransferas...,"SET domain superfamily | - | SMYD1, SET domain..."
2352,ENSG00000151914_ProteinCoding_4,-0.525019250063025,True,DST,ENST00000680361,chr6,56529448,56529774,327,327,7818,True,7,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,- | microtubule-actin cross-linking factor 1 |...,- | Spectrin/alpha-actinin | Spectrin repeat
3105,ENSG00000176884_ProteinCoding_1,0.3713913965532838,True,GRIN1,ENST00000371560,chr9,137148152,137148214,63,63,906,True,6,CATH-FunFam | CDD | PIRSR | Pfam | Phobius | S...,"glutamate receptor ionotropic, NMDA 1 isoform ...","- | Glutamate [NMDA] receptor subunit 1-like, ..."
3627,ENSG00000243955_ProteinCoding_1,0.3461300290715827,True,GSTA1,ENST00000334575,chr6,52796182,52796314,133,133,222,False,6,CATH-Gene3D | CDD | SUPERFAMILY | CATH-FunFam ...,"- | Glutaredoxin | C-terminal, alpha helical d...",- | Thioredoxin-like superfamily | Glutathione...
2197,ENSG00000146122_ProteinCoding_1,0.2245671444135169,True,DAAM2,ENST00000633794,chr6,39886421,39886447,27,27,1077,True,6,CATH-FunFam | Pfam | SMART | SUPERFAMILY | PRO...,Dishevelled associated activator of morphogene...,"- | Formin, FH2 domain | Formin, FH2 domain su..."


## Visualize interesting hits

In [ ]:
import pickle
from collections import Counter
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
"""
plot_clusters.py

Genome-browser-style visualization of one splicing cluster.
Left panel  : exon/intron structure per event.
Right panel : per-cell-type r-value heatmap (from signif_info).

Usage
-----
from plot_clusters import plot_cluster

plot_cluster(cid, members[cid], event_info, signif_info)
plot_cluster(cid, members[cid], event_info, signif_info, save_path='cluster_5178.pdf')

signif_info format
------------------
signif_info[event_id]['gene_name'] = str                                  # store once per event
signif_info[event_id][cell_type]   = {'r': float, 'fdr': float, 'is_specific': bool}

Build it like this:
    for idx, row in signif_exons_df.iterrows():
        signif_info[idx]['gene_name'] = row['Gene']
        signif_info[idx][ct] = {'is_specific': row['is_specific'],
                                 'r': row['r'], 'fdr': row['fdr']}
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm


# ── shared row builder (both panels use the same sorted order) ─────────────
def _build_rows(event_ids, event_info, signif_set):
    rows = []
    for ev in event_ids:
        if ev not in event_info:
            continue
        meta = event_info[ev]['meta']
        rows.append({
            'event': ev,
            'chrom': meta['chrom'],
            'gene':  meta.get('gene', ''),
            'es':    meta['es'],
            'ee':    meta['ee'],
            'us':    meta['us_intron_start'],
            'ds':    meta['ds_intron_end'],
            'signif': ev in signif_set,
        })
    rows.sort(key=lambda r: (r['es'], r['ee'], r['ds']))
    return rows


# ── left panel: exon / intron structure ───────────────────────────────────
def _plot_structure(ax, cid, rows, signif_set, signif_info):
    if not rows:
        ax.set_visible(False)
        return

    x_min = min(r['us'] for r in rows)
    x_max = max(r['ds'] for r in rows)
    span  = max(x_max - x_min, 1)
    pad   = span * 0.04

    for i, r in enumerate(rows):
        color = 'tomato' if r['signif'] else 'steelblue'

        # intron span line
        ax.plot([r['us'], r['ds']], [i, i],
                color='#bbbbbb', lw=0.8, zorder=1, solid_capstyle='round')

        # cassette exon block
        width = max(r['ee'] - r['es'], span * 0.003)
        ax.add_patch(Rectangle((r['es'], i - 0.3), width, 0.6,
                                facecolor=color, edgecolor='none', zorder=2))

        # event label
        parts  = r['event'].split('_')
        label  = '_'.join(parts[-2:]) if len(parts) >= 2 else r['event']
        marker = ' ★' if r['signif'] else ''
        ax.text(x_min - pad, i, label + marker,
                ha='right', va='center', fontsize=5.5,
                color='tomato' if r['signif'] else '#555555')

    # gene display name: scan all rows for first event that has gene_name in signif_info
    gene_display = next(
        (signif_info[r['event']].get('gene_name')
        for r in rows
        if r['event'] in signif_info and signif_info[r['event']].get('gene_name')),
        rows[0]['gene']
    )
    n_sig = sum(r['signif'] for r in rows)
    ax.set_title(
        f"cluster {cid}  |  {gene_display}  |  {rows[0]['chrom']}\n"
        f"{len(rows)} events  ({n_sig} significant in ≥1 cell type)",
        fontsize=6.5, loc='left', pad=3
    )
    ax.set_xlim(x_min - pad * 2, x_max + pad)
    ax.set_ylim(-0.8, len(rows) - 0.2)
    ax.set_yticks([])

    def _kb(x, _):
        return f'+{(x - x_min) / 1000:.1f}kb' if x != x_min else '0'
    ax.xaxis.set_major_formatter(plt.FuncFormatter(_kb))
    ax.xaxis.set_major_locator(plt.MaxNLocator(4, integer=False))
    ax.tick_params(axis='x', labelsize=5.5, pad=1)
    for spine in ('top', 'left', 'right'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)


# ── right panel: per-cell-type r-value heatmap ────────────────────────────
def _plot_celltype_panel(ax, rows, signif_info, cell_types):
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    cmap = plt.get_cmap('RdBu_r')

    for i, r in enumerate(rows):
        ev = r['event']
        for j, ct in enumerate(cell_types):
            info = signif_info.get(ev, {}).get(ct)
            if isinstance(info, dict):
                color = cmap(norm(float(np.clip(info['r'], -1, 1))))
                ec = 'black' if info.get('is_specific') else 'white'
                lw = 1.2    if info.get('is_specific') else 0.3
            else:
                color, ec, lw = '#eeeeee', 'white', 0.3

            ax.add_patch(Rectangle((j, i - 0.4), 1, 0.8,
                                    facecolor=color, edgecolor=ec,
                                    linewidth=lw, zorder=2))

            # dot for FDR < 0.01
            if isinstance(info, dict) and info.get('fdr', 1) < 0.01:
                ax.text(j + 0.5, i, '·', ha='center', va='center',
                        fontsize=9, color='white', zorder=3)

    ax.set_xlim(0, len(cell_types))
    ax.set_ylim(-0.8, len(rows) - 0.2)
    ax.set_xticks([j + 0.5 for j in range(len(cell_types))])
    ax.set_xticklabels(cell_types, rotation=40, ha='left', fontsize=5.5)
    ax.xaxis.set_tick_params(length=0)
    ax.tick_params(axis='x', pad=1)
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    # colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02, aspect=12)
    cbar.set_label('r', fontsize=6)
    cbar.ax.tick_params(labelsize=5)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])


# ── main entry point ───────────────────────────────────────────────────────
def plot_cluster(cid, event_ids, event_info, signif_info,
                 figsize=None, save_path=None, dpi=150):
    """
    Parameters
    ----------
    cid           : cluster id (for the title)
    event_ids     : members[cid]
    event_info     : dict  event_id -> annotate_event record
    signif_info   : dict  event_id -> {'gene_name': str, cell_type: {'r','fdr','is_specific'}}
    figsize       : (width, height) in inches; None = auto
    save_path     : file path to save (e.g. 'cluster.pdf'); None = plt.show()
    dpi           : resolution for raster output
    """
    signif_set = set(signif_info.keys())
    rows       = _build_rows(event_ids, event_info, signif_set)

    if not rows:
        print(f"cluster {cid}: no events found in event_info")
        return None

    # cell types: all keys except 'gene_name', skip non-dict values defensively
    cell_types = sorted({
        ct
        for r in rows
        for ct, val in signif_info.get(r['event'], {}).items()
        if ct != 'gene_name' and isinstance(val, dict)
    })

    n      = len(rows)
    height = max(1.8, n * 0.38)

    if cell_types:
        n_ct         = len(cell_types)
        default_size = (7 + n_ct * 0.7, height)
        fig, (ax_l, ax_r) = plt.subplots(
            1, 2,
            figsize=figsize or default_size,
            gridspec_kw={'width_ratios': [4, max(1, n_ct)]}
        )
        _plot_celltype_panel(ax_r, rows, signif_info, cell_types)
        ax_r.set_title('cell-type associations\n(border = specific, · = FDR<0.01)',
                        fontsize=6, loc='left', pad=3)
    else:
        fig, ax_l = plt.subplots(figsize=figsize or (7, height))

    _plot_structure(ax_l, cid, rows, signif_set, signif_info)

    ax_l.legend(handles=[
        mpatches.Patch(facecolor='tomato',    label='significant (any ct)'),
        mpatches.Patch(facecolor='steelblue', label='not significant'),
    ], loc='upper right', fontsize=5.5, frameon=False)

    plt.tight_layout(pad=1.5)

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"saved -> {save_path}")
    else:
        plt.show()

    return fig

In [ ]:
event_dicts = pickle.load(open('event_dicts.pkl', 'rb'))
event_info = pickle.load(open('event_info.pkl', 'rb'))

In [ ]:
# Get all significant splicing events
signif_info = defaultdict(dict)
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        ct = file.replace("_exons.csv", "")
        for idx, row in signif_exons_df.iterrows():
            signif_info[idx]['gene_name'] = row['Gene']   # store once at event level
            signif_info[idx][ct] = {'is_specific': row['is_specific'], 'r': row['r'], 'fdr': row['fdr']}

In [ ]:
members = defaultdict(list)
for ed in event_dicts:
    members[ed['cluster_id']].append(ed['event'])

In [ ]:
sizes = Counter(ed['cluster_id'] for ed in event_dicts)

specific_cids = [
    cid for cid, evs in members.items()
    if any(
        any(info['is_specific'] for info in signif_info.get(ev, {}).values()
            if isinstance(info, dict))
        for ev in evs
    )
]

multi = {cid: n for cid, n in sizes.items() if (n > 1) and cid in specific_cids}
sorted_cids = [cid for cid, n in sorted(multi.items(), key=lambda kv: -kv[1])]

In [ ]:
with PdfPages('all_clusters.pdf') as pdf:
    for cid in sorted_cids[:50]:
        print(cid)
        fig = plot_cluster(cid, members[cid], event_info, signif_info)
        if fig is not None:
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)